In [1]:
import sys
from pathlib import Path
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import copy
import matplotlib.pyplot as plt

def find_project_root(start=None):
    if start is None:
        start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    return start


PROJECT_ROOT = find_project_root()
sys.path.append(str(PROJECT_ROOT))



In [2]:
from src.io.pitch_io import load_pitch_file
from src.io.annotation_io import load_annotations
import settings as S

START_COL = "start_time_sec"
END_COL = "end_time_sec"



This block defines the parameters that control how pitch contours are converted into training data.

Key parameters:

- `WINDOW_SIZE`: number of samples per training window
- `STRIDE`: distance between consecutive windows
- `MIN_RUN_LEN`: minimum number of valid samples required for a contour fragment
- `NORMALIZE_PER_WINDOW`: whether each window is z-score normalized

These parameters determine how much local context the model sees and how many training examples are generated.

In [3]:
SEED = 42

WINDOW_SIZE = 32
MIN_RUN_LEN = WINDOW_SIZE

STRIDE = 4

NORMALIZE_PER_WINDOW = True

# data split
VAL_RATIO = 0.20

# masking
MIN_MASK_LEN = 4
MAX_MASK_LEN = 6
MASK_FILL_VALUE = 0.0
USE_MASK_CHANNEL = True

# dataloader
BATCH_SIZE = 16

# model
HIDDEN_CHANNELS = 64
DROPOUT = 0.05
IN_CHANNELS = 2 if USE_MASK_CHANNEL else 1

# training
NUM_EPOCHS = 150
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

# reproducibility
SEED = 999


In [4]:
def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

Each training window can optionally be normalized using z-score normalization.

This transforms the window so that:

- mean = 0
- standard deviation = 1

Normalization helps the model focus on the **shape of the pitch trajectory** rather than absolute pitch height.

Since the pitch is already expressed in cents relative to the tonic, normalization mainly removes local offset differences.

In [5]:
def zscore_normalize(x, eps=1e-8):
    mean = x.mean()
    std = x.std()

    if std < eps:
        return x - mean

    return (x - mean) / (std + eps)

Long pitch contours are split into fixed-length windows.

Given a contour:

- windows of length `WINDOW_SIZE` are extracted
- consecutive windows are separated by `STRIDE` samples

This produces overlapping windows and increases the number of training examples.

For example, a contour of 1000 samples can produce dozens of training windows.

In [6]:
def slice_windows(contour, window_size=256, stride=64):
    windows = []
    n = len(contour)

    if n < window_size:
        return windows

    for start in range(0, n - window_size + 1, stride):
        end = start + window_size
        windows.append(contour[start:end])

    return windows


def build_window_corpus(clean_contours,
                        window_size=32,
                        stride=8,
                        normalize_per_window=True):
    all_windows = []

    for contour in clean_contours:
        windows = slice_windows(contour, window_size=window_size, stride=stride)

        for w in windows:
            w = np.asarray(w, dtype=np.float32)

            if normalize_per_window:
                w = zscore_normalize(w)

            all_windows.append(w.astype(np.float32))

    if len(all_windows) == 0:
        raise ValueError("No windows were created. Check MIN_RUN_LEN and WINDOW_SIZE.")

    return np.stack(all_windows, axis=0)

A boolean mask is constructed to identify pitch samples that are considered reliable.

This mask uses information already computed during preprocessing, such as:

- `too_long_to_interp`
- `is_outlier`
- `valid_for_pchip`

Only samples that pass these checks are considered valid.

Using these existing flags avoids repeating preprocessing steps and ensures consistency with the pitch cleaning pipeline.

In [7]:
def build_valid_sample_mask(df_pitch, pitch_col):
    pitch_vals = df_pitch[pitch_col].to_numpy()
    valid_mask = np.isfinite(pitch_vals)

    if "too_long_to_interp" in df_pitch.columns:
        valid_mask &= ~df_pitch["too_long_to_interp"].to_numpy()

    if "is_outlier" in df_pitch.columns:
        valid_mask &= ~df_pitch["is_outlier"].to_numpy()

    if "valid_for_pchip" in df_pitch.columns:
        valid_mask &= df_pitch["valid_for_pchip"].to_numpy()

    return valid_mask

Within each segment, the code identifies contiguous sequences of valid samples.

Instead of removing invalid samples and compressing the contour, the algorithm detects **runs of consecutive valid points**.

Each run represents a temporally continuous portion of the pitch trajectory.

This prevents two valid fragments separated by invalid samples from being artificially merged into a single contour.

In [8]:

def find_true_runs(mask):
    """
    Returns a list of (start_idx, end_idx) pairs for contiguous True runs.
    end_idx is exclusive.
    """
    runs = []
    n = len(mask)

    i = 0
    while i < n:
        if mask[i]:
            start = i
            while i < n and mask[i]:
                i += 1
            end = i
            runs.append((start, end))
        else:
            i += 1

    return runs


Pitch contours are extracted for each annotated svara segment.

For every annotation:

1. The corresponding time interval is selected from the pitch dataframe.
2. The validity mask is applied.
3. Continuous valid runs are detected.
4. Runs shorter than `MIN_RUN_LEN` are discarded.

Each remaining run is stored as an independent pitch contour.

This produces a collection of clean pitch fragments that can be used for training.

In [9]:
def extract_segment_contours(df_pitch,
                             df_svaras,
                             pitch_col,
                             min_run_len=32):
    time_vals = df_pitch[S.TIME_COL].to_numpy()
    pitch_vals = df_pitch[pitch_col].to_numpy()

    valid_mask_global = build_valid_sample_mask(df_pitch, pitch_col)

    starts = df_svaras[START_COL].to_numpy()
    ends = df_svaras[END_COL].to_numpy()

    clean_contours = []

    for start_sec, end_sec in zip(starts, ends):
        seg_mask = (time_vals >= start_sec) & (time_vals <= end_sec)

        seg_pitch = pitch_vals[seg_mask]
        seg_valid = valid_mask_global[seg_mask]

        runs = find_true_runs(seg_valid)

        for start_idx, end_idx in runs:
            contour = seg_pitch[start_idx:end_idx]

            if len(contour) < min_run_len:
                continue

            contour = np.asarray(contour, dtype=np.float32)

            if not np.all(np.isfinite(contour)):
                continue

            clean_contours.append(contour)

    if len(clean_contours) == 0:
        raise ValueError("No valid continuous contours were extracted.")

    return clean_contours

In [10]:
def hz_to_cents(pitch_hz, tonic_hz):
    """
    Convert pitch in Hz to cents relative to a tonic.

    pitch_hz : array-like
    tonic_hz : float
    """

    pitch_hz = np.asarray(pitch_hz, dtype=np.float64)

    cents = np.full_like(pitch_hz, np.nan, dtype=np.float64)

    valid = np.isfinite(pitch_hz) & (pitch_hz > 0)

    cents[valid] = 1200 * np.log2(pitch_hz[valid] / tonic_hz)

    return cents

Two datasets are used:

- the **preprocessed pitch trajectory** of the recording
- the **svara annotation file**

The pitch is loaded with the option `convert_to_cents=True`, which converts frequencies from Hz to cents relative to the tonic.

Working in cents makes pitch trajectories comparable across recordings with different tonic frequencies.

In [14]:
set_seed(SEED)

recording_id = S.CURRENT_PIECE
tonic_hz = S.SARASUDA_TONICS[recording_id]
pitch_path = S.DATA_INTERIM / recording_id / "pitch" / f"{recording_id}_pitch_preprocessed_debug.parquet"

annotation_path = (
    S.DATA_CORPUS
    / recording_id
    / "raw"
    / f"{recording_id}_ann_svara.tsv"
)


df_pitch = load_pitch_file(
    file_path=pitch_path
)

df_svaras = load_annotations(
    file_path=annotation_path,
    annotation_type="svara",
    engine="polars",
    
)

pitch_hz = df_pitch[S.PITCH_COL_RAW].to_numpy()


PITCH_COL_RAW_CENTS = hz_to_cents(pitch_hz = pitch_hz, tonic_hz = S.SARASUDA_TONICS[recording_id])

pitch_cents = hz_to_cents(
    pitch_hz=df_pitch[S.PITCH_COL_RAW].to_numpy(),
    tonic_hz=S.SARASUDA_TONICS[recording_id]
)

PITCH_COL_RAW_CENTS_NAME = "f0_raw_cents"

df_pitch = df_pitch.with_columns(
    pl.Series(PITCH_COL_RAW_CENTS_NAME, pitch_cents)
)




Using the previously defined functions, continuous pitch contours are extracted from the annotated segments.

Each contour corresponds to a continuous region of valid pitch samples within a svara segment.

The result is a list of 1D NumPy arrays, where each array represents a clean pitch trajectory fragment.

In [ ]:
BASE_MIN_RUN_LEN = 32

clean_contours = extract_segment_contours(
    df_pitch,
    df_svaras=df_svaras,
    pitch_col=PITCH_COL_RAW_CENTS_NAME,
    min_run_len=BASE_MIN_RUN_LEN,
)

print("Number of clean continuous contours:", len(clean_contours))
print("First contour length:", len(clean_contours[0]))

TypeError: cannot select columns using NumPy array of type float64

The extracted contours are finally converted into the training corpus.

Each contour is split into overlapping windows using the parameters:

- `WINDOW_SIZE`
- `STRIDE`

The result is a matrix of shape: [N_windows, WINDOW_SIZE] where each row corresponds to one training example.

The next step is to divide the window corpus into two subsets:

- a **training set**, used to optimize the model parameters
- a **validation set**, used to monitor generalization during training

The split is done at the window level.  
This is sufficient for a first experiment, although later it may be useful to split by larger musical units to make validation stricter.

A random permutation is applied before the split so that the two subsets are not biased by the original ordering of the contours.

In [ ]:
def train_val_split(windows, val_ratio=0.2, seed=42):
    n = len(windows)

    idx = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)

    n_val = int(n * val_ratio)

    val_idx = idx[:n_val]
    train_idx = idx[n_val:]

    train_windows = windows[train_idx]
    val_windows = windows[val_idx]

    return train_windows, val_windows


This module builds the dataset used for masked reconstruction.

For each input window:

1. a contiguous span is selected at random
2. that span is masked
3. the model receives the masked window as input
4. the original window is kept as the target
5. a binary mask indicates where the reconstruction loss should be computed

The masking is self-supervised because the target is taken directly from the original signal itself.

A contiguous mask is more appropriate than random isolated points because pitch contours are continuous temporal trajectories.

In [ ]:

class PitchMaskingDataset(Dataset):
    def __init__(self,
                 windows,
                 min_mask_len=4,
                 max_mask_len=8,
                 use_mask_channel=True,
                 mask_fill_value=0.0):
        self.windows = windows
        self.min_mask_len = min_mask_len
        self.max_mask_len = max_mask_len
        self.use_mask_channel = use_mask_channel
        self.mask_fill_value = mask_fill_value

    def __len__(self):
        return len(self.windows)

    def _sample_mask(self, length):
        mask = np.zeros(length, dtype=np.float32)

        mask_len = np.random.randint(self.min_mask_len, self.max_mask_len + 1)
        mask_len = min(mask_len, length)

        start = np.random.randint(0, length - mask_len + 1)
        end = start + mask_len

        mask[start:end] = 1.0
        return mask

    def _apply_mask(self, x, mask):
        x_masked = x.copy()
        x_masked[mask == 1] = self.mask_fill_value
        return x_masked

    def __getitem__(self, idx):
        x = self.windows[idx].astype(np.float32)

        mask = self._sample_mask(len(x))
        x_masked = self._apply_mask(x, mask)

        if self.use_mask_channel:
            inp = np.stack([x_masked, mask], axis=0)   # [2, L]
        else:
            inp = x_masked[None, :]                    # [1, L]

        target = x[None, :]                            # [1, L]
        mask = mask[None, :]                           # [1, L]

        return {
            "input": torch.tensor(inp, dtype=torch.float32),
            "target": torch.tensor(target, dtype=torch.float32),
            "mask": torch.tensor(mask, dtype=torch.float32),
        }



The input to the model contains two channels:

- the masked pitch window
- the binary mask

The first channel is the actual pitch trajectory after replacing the masked region with zeros.  
The second channel explicitly tells the model which samples are hidden.

This makes the task clearer for the model, since it does not need to infer by itself where information is missing.

Each dataset item returns three tensors:

- `input`: shape `[C, L]`
- `target`: shape `[1, L]`
- `mask`: shape `[1, L]`

where:

- `L` is the window length
- `C` is the number of input channels

These tensors will be used in the next module to train the convolutional reconstruction model.

The model receives a short pitch window and predicts a reconstructed version of the same window.

The architecture is intentionally simple:

- 1D convolutions
- local nonlinear transformations
- same input and output length

This is appropriate because pitch contours are one-dimensional temporal signals, and the main goal is to model local continuity and shape.


The input tensor has shape:

[B, C, L] 


where:

B is the batch size
C is the number of channels
L is the window length


If the mask channel is used, then:

channel 1 = masked pitch contour
channel 2 = binary mask

The model output has shape: [B, 1, L]
so that one reconstructed pitch value is predicted for each sample in the window.

In [ ]:
class ConvMaskedAutoencoder(nn.Module):
    def __init__(self, in_channels=2, hidden_channels=32, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(in_channels, hidden_channels, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(hidden_channels, hidden_channels, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(hidden_channels, hidden_channels * 2, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(hidden_channels * 2, hidden_channels, kernel_size=5, padding=2),
            nn.ReLU(),

            nn.Conv1d(hidden_channels, 1, kernel_size=1)
        )

    def forward(self, x):
        return self.net(x)

The loss is computed only on the masked region.

This is important because the visible part of the input is already known to the model.  
If the loss were computed everywhere, the task would become too easy and the network could minimize the error by copying the visible samples.

Using a masked loss forces the model to infer the hidden fragment from its local context.

In [ ]:
def masked_mse_loss(pred, target, mask, eps=1e-8):
    """
    pred   : [B, 1, L]
    target : [B, 1, L]
    mask   : [B, 1, L]
    """
    sq_error = (pred - target) ** 2
    masked_sq_error = sq_error * mask
    loss = masked_sq_error.sum() / (mask.sum() + eps)
    return loss

In [ ]:

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

This module trains the convolutional model to reconstruct masked pitch windows.

At each iteration:

1. the model receives a masked input window
2. it predicts a reconstructed contour
3. the masked reconstruction loss is computed
4. the parameters are updated with backpropagation

The objective is not to reconstruct the full visible signal, but specifically to infer the hidden region from the available local context.

In [ ]:
def run_epoch(model, loader, optimizer=None, device="cpu"):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        x = batch["input"].to(device)
        y = batch["target"].to(device)
        m = batch["mask"].to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            pred = model(x)
            loss = masked_mse_loss(pred, y, m)

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    mean_loss = total_loss / max(n_batches, 1)
    return mean_loss

Two loops are used:

- a **training loop**, where model parameters are updated
- a **validation loop**, where the model is evaluated without updating parameters

The validation loss is used to monitor generalization and detect whether the model is improving beyond the training set.

During training, the model state with the lowest validation loss is stored.

At the end of training, the best validation checkpoint is restored.

This is a simple and effective way to keep the most useful model, even if later epochs fluctuate or begin to overfit.

After training the model, the next step is to inspect its predictions.

The goal is to verify whether the model has learned to reconstruct masked pitch regions using the surrounding context.

This inspection is qualitative: we visualize the predicted pitch contour together with the original contour and the masked input.

Since pitch contours are smooth temporal trajectories, visual comparison is a very informative diagnostic tool.

In [ ]:
def reconstruct_window(model, dataset, index, device="cpu"):

    item = dataset[index]

    x = item["input"].unsqueeze(0).to(device)
    target = item["target"].squeeze().numpy()
    mask = item["mask"].squeeze().numpy()

    model.eval()

    with torch.no_grad():
        pred = model(x).cpu().squeeze().numpy()

    if USE_MASK_CHANNEL:
        x_masked = item["input"][0].numpy()
    else:
        x_masked = item["input"].squeeze().numpy()

    return x_masked, pred, target, mask


In [ ]:
def plot_reconstruction(x_masked, pred, target, mask):

    L = len(target)
    x_axis = np.arange(L)

    plt.figure(figsize=(10,4))

    plt.plot(x_axis, target, label="target", linewidth=2)
    plt.plot(x_axis, x_masked, label="masked input", linestyle="--")
    plt.plot(x_axis, pred, label="prediction", linewidth=2)

    mask_idx = np.where(mask == 1)[0]

    if len(mask_idx) > 0:
        plt.scatter(mask_idx, target[mask_idx], color="red", s=20, label="masked region")

    plt.legend()
    plt.xlabel("sample")
    plt.ylabel("pitch")
    plt.title("Masked pitch reconstruction")

    plt.show()


In [ ]:
from itertools import product
import pandas as pd


SEARCH_SPACE = {
    "window_size": [32],
    "stride": [4],
    "min_mask_len": [2],
    "max_mask_len": [4],
    "batch_size": [16],
    "hidden_channels": [96],
    "dropout": [0.05],
    "learning_rate": [5e-4],
    "num_epochs": [150],
}

SEEDS_TO_TRY = [42, 123, 999]

In [ ]:
def build_config_list(search_space):
    keys = list(search_space.keys())
    values = [search_space[k] for k in keys]

    configs = []
    for combo in product(*values):
        cfg = dict(zip(keys, combo))
        cfg["min_run_len"] = cfg["window_size"]
        configs.append(cfg)

    return configs


grid_configs = build_config_list(SEARCH_SPACE)

print("Number of configurations:", len(grid_configs))
for i, cfg in enumerate(grid_configs):
    print(i, cfg)

In [ ]:
def run_single_experiment(clean_contours, cfg, seed=42):
    set_seed(seed)

    windows = build_window_corpus(
        clean_contours=clean_contours,
        window_size=cfg["window_size"],
        stride=cfg["stride"],
        normalize_per_window=NORMALIZE_PER_WINDOW,
    )

    train_windows, val_windows = train_val_split(
        windows,
        val_ratio=VAL_RATIO,
        seed=seed,
    )

    train_dataset = PitchMaskingDataset(
        windows=train_windows,
        min_mask_len=cfg["min_mask_len"],
        max_mask_len=cfg["max_mask_len"],
        use_mask_channel=USE_MASK_CHANNEL,
        mask_fill_value=MASK_FILL_VALUE,
    )

    val_dataset = PitchMaskingDataset(
        windows=val_windows,
        min_mask_len=cfg["min_mask_len"],
        max_mask_len=cfg["max_mask_len"],
        use_mask_channel=USE_MASK_CHANNEL,
        mask_fill_value=MASK_FILL_VALUE,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg["batch_size"],
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg["batch_size"],
        shuffle=False,
    )

    in_channels = 2 if USE_MASK_CHANNEL else 1

    model = ConvMaskedAutoencoder(
        in_channels=in_channels,
        hidden_channels=cfg["hidden_channels"],
        dropout=cfg["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["learning_rate"],
        weight_decay=WEIGHT_DECAY,
    )

    best_val_loss = float("inf")
    best_epoch = None
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, cfg["num_epochs"] + 1):
        train_loss = run_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=DEVICE,
        )

        val_loss = run_epoch(
            model=model,
            loader=val_loader,
            optimizer=None,
            device=DEVICE,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)

    return {
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "n_windows": len(windows),
        "n_train_windows": len(train_windows),
        "n_val_windows": len(val_windows),
        "best_model_state": best_state,
    }

In [ ]:
results = []
best_model_state = None
best_overall_val_loss = float("inf")
best_config = None

for cfg_id, cfg in enumerate(grid_configs):
    print(f"\n=== CONFIG {cfg_id + 1}/{len(grid_configs)} ===")
    print(cfg)

    for seed in SEEDS_TO_TRY:
        print(f"  seed={seed}")

        out = run_single_experiment(
            clean_contours=clean_contours,
            cfg=cfg,
            seed=seed,
        )

        row = {
            "config_id": cfg_id,
            "seed": seed,
            **cfg,
            "best_val_loss": out["best_val_loss"],
            "best_epoch": out["best_epoch"],
            "n_windows": out["n_windows"],
            "n_train_windows": out["n_train_windows"],
            "n_val_windows": out["n_val_windows"],
        }

        results.append(row)

        print(
            f"    best_val_loss={out['best_val_loss']:.6f} | "
            f"best_epoch={out['best_epoch']}"
        )

        if out["best_val_loss"] < best_overall_val_loss:
            best_overall_val_loss = out["best_val_loss"]
            best_model_state = out["best_model_state"]
            best_config = row.copy()

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    ["best_val_loss", "config_id", "seed"]
).reset_index(drop=True)

results_df.head(20)

In [ ]:
summary_df = (
    results_df
    .groupby(
        [
            "config_id",
            "window_size",
            "stride",
            "min_mask_len",
            "max_mask_len",
            "batch_size",
            "hidden_channels",
            "dropout",
            "learning_rate",
            "num_epochs",
        ],
        as_index=False
    )
    .agg(
        mean_best_val_loss=("best_val_loss", "mean"),
        std_best_val_loss=("best_val_loss", "std"),
        min_best_val_loss=("best_val_loss", "min"),
        max_best_val_loss=("best_val_loss", "max"),
        mean_best_epoch=("best_epoch", "mean"),
    )
    .sort_values("mean_best_val_loss")
    .reset_index(drop=True)
)

summary_df.head(20)

In [ ]:
print("Best overall configuration:")
print(best_config)

best_model = ConvMaskedAutoencoder(
    in_channels=2 if USE_MASK_CHANNEL else 1,
    hidden_channels=best_config["hidden_channels"],
    dropout=best_config["dropout"],
).to(DEVICE)

best_model.load_state_dict(best_model_state)
best_model.eval()

In [ ]:
best_windows = build_window_corpus(
    clean_contours=clean_contours,
    window_size=best_config["window_size"],
    stride=best_config["stride"],
    normalize_per_window=NORMALIZE_PER_WINDOW,
)

best_train_windows, best_val_windows = train_val_split(
    best_windows,
    val_ratio=VAL_RATIO,
    seed=best_config["seed"],
)

best_val_dataset = PitchMaskingDataset(
    windows=best_val_windows,
    min_mask_len=best_config["min_mask_len"],
    max_mask_len=best_config["max_mask_len"],
    use_mask_channel=USE_MASK_CHANNEL,
    mask_fill_value=MASK_FILL_VALUE,
)

N_EXAMPLES = 5

for i in range(N_EXAMPLES):
    idx = np.random.randint(len(best_val_dataset))

    x_masked, pred, target, mask = reconstruct_window(
        best_model,
        best_val_dataset,
        idx,
        device=DEVICE
    )

    plot_reconstruction(x_masked, pred, target, mask)

In [ ]:
import json
from datetime import datetime

# --------------------------------------------------
# Create experiment directory with timestamp
# --------------------------------------------------

experiment_name = "pitch_masked_autoencoder_savgol13"

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

results_dir = PROJECT_ROOT / "logs" / experiment_name / timestamp
results_dir.mkdir(parents=True, exist_ok=True)

print("Saving experiment to:")
print(results_dir)


# --------------------------------------------------
# Save tables
# --------------------------------------------------

results_df.to_parquet(results_dir / "results_full.parquet")
summary_df.to_parquet(results_dir / "results_summary.parquet")

print("Saved results tables")


# --------------------------------------------------
# Save best configuration
# --------------------------------------------------

with open(results_dir / "best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)

print("Saved best_config.json")


# --------------------------------------------------
# Save experiment configuration
# --------------------------------------------------

config_used = {
    "search_space": SEARCH_SPACE,
    "seeds": SEEDS_TO_TRY,
    "normalize_per_window": NORMALIZE_PER_WINDOW,
    "val_ratio": VAL_RATIO,
    "mask_fill_value": MASK_FILL_VALUE,
    "use_mask_channel": USE_MASK_CHANNEL,
    "weight_decay": WEIGHT_DECAY,
    "experiment_timestamp": timestamp
}

with open(results_dir / "config_used.json", "w") as f:
    json.dump(config_used, f, indent=2)

print("Saved config_used.json")


# --------------------------------------------------
# Save best model
# --------------------------------------------------
e
torch.save(best_model_state, results_dir / "best_model.pt")

print("Saved best_model.pt")


print("\nExperiment successfully stored.")